In [15]:
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [16]:
data_dir = r"C:\Users\Poojani Danulya\Desktop\ML(JN)\CropCortex\PlantVillage"

In [17]:
train_dir = os.path.join(data_dir, "train")

In [18]:
classes = os.listdir(train_dir)
print(classes)
print("Number of classes:", len(classes))

['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___healthy', 'Corn_(maize)___Northern_Leaf_Blight', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___healthy', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___healthy', 'Potato___Late_blight', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___healthy', 'Strawberry___Leaf_scorch', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___healthy', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_mite',

## Split training dataset into train and test datasets

import os
import shutil
import random

data_dir = r"C:\Users\Poojani Danulya\Desktop\ML(JN)\CropCortex\PlantVillage"

train_dir = os.path.join(data_dir, "train")
test_dir = os.path.join(data_dir, "test")

# create test directory
os.makedirs(test_dir, exist_ok=True)

split_ratio = 0.2  # 20% for test

for class_name in os.listdir(train_dir):
    class_path = os.path.join(train_dir, class_name)

    if not os.path.isdir(class_path):
        continue

    # create class folder in test
    test_class_path = os.path.join(test_dir, class_name)
    os.makedirs(test_class_path, exist_ok=True)

    # get all images in class
    images = os.listdir(class_path)

    # shuffle images
    random.shuffle(images)

    # split index
    split_index = int(len(images) * split_ratio)

    test_images = images[:split_index]

    # move images to test folder
    for img in test_images:
        src = os.path.join(class_path, img)
        dst = os.path.join(test_class_path, img)

        shutil.move(src, dst)

print("Dataset split completed!")

## DEFINE IMAGE SETTINGS

In [21]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

## DATA AUGMENTATION (TRAIN ONLY)

In [22]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

## VALIDATION & TEST DATA (NO AUGMENTATION)

In [23]:
val_test_datagen = ImageDataGenerator(rescale=1./255)

In [26]:
val_dir = r"C:\Users\Poojani Danulya\Desktop\ML(JN)\CropCortex\PlantVillage\val"
test_dir = r"C:\Users\Poojani Danulya\Desktop\ML(JN)\CropCortex\PlantVillage\test"

## LOAD DATA

In [27]:
train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

Found 34771 images belonging to 38 classes.


In [28]:
val_data = val_test_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

Found 10861 images belonging to 38 classes.


In [29]:
test_data = val_test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

Found 8673 images belonging to 38 classes.


In [31]:
for x,y in train_data:
    print(x, x.shape)
    print(y, y.shape)
    break

[[[[0.5176471  0.4784314  0.58431375]
   [0.5176471  0.4784314  0.58431375]
   [0.5176471  0.4784314  0.58431375]
   ...
   [0.5487169  0.51734436 0.611462  ]
   [0.6173184  0.58594584 0.6800635 ]
   [0.5777266  0.54635406 0.64047176]]

  [[0.5374537  0.49823803 0.6041204 ]
   [0.5316563  0.49244064 0.598323  ]
   [0.5258589  0.48664322 0.59252554]
   ...
   [0.5642661  0.53289354 0.6270112 ]
   [0.61123335 0.5798608  0.6739785 ]
   [0.5702276  0.5388551  0.6329727 ]]

  [[0.55580753 0.51659185 0.6224742 ]
   [0.5551634  0.5159477  0.62183005]
   [0.55451924 0.5153035  0.6211859 ]
   ...
   [0.5798152  0.54844266 0.6425603 ]
   [0.6014129  0.57004035 0.664158  ]
   [0.56531733 0.5339448  0.6280624 ]]

  ...

  [[0.3320929  0.48895565 0.29287723]
   [0.32120332 0.4780661  0.28198764]
   [0.3270503  0.48391303 0.2878346 ]
   ...
   [0.559972   0.5089916  0.57565826]
   [0.5580395  0.5070591  0.57372576]
   [0.55610704 0.5051266  0.5717933 ]]

  [[0.31163353 0.4684963  0.27241784]
   [0.3

x.shape = (32, 224, 224, 3) means 32 images in 1 batch (1 input contained 32 images) with 224 x 224 height x width and 3 color channels RGB.

## IMPORT MobileNetV2

In [33]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import models, layers

## LOAD PRETRAINED MODEL

In [34]:
base_model = MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights="imagenet"
)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 8s 1us/step


## FREEZ THE BASE MODEL

In [35]:
base_model.trainable = False

We freeze the model to protect its pretrained knowledge and train only our classifier first

## ADDING THE CUSTOM LAYERS OF MY CLASSIFIER

In [37]:
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(38, activation='softmax')
])

1. Input 32 images passes through base_model, dozens of convolution layers detecting edges, textures and patterns.
2. Output of MobileNetV2 is a group of values called Feature Maps with 1000+ feature detectors each detecting a specific feature. (Extract Features)
3. Feature Maps go into GlobalAveragePooling2D converting (7, 7, 1280) → (1280). It takes average of each feature map and reduce features. (Compress)
4. Then in 1st Dense Layer ==> Input:1280 features -> Output:128 neurons. There model learns disease patterns with feature interpretation.
5. Then those 1280 neurones pass to Dropout(0.3) where randomly turns OFF 30% of neurons to prevent onerfitting.
6. Then in last Dense Layer converts outputs into probabilities.